# Design Spec — Rust (evcxr) + Go (gonb) Kernels with Ports Parity

**Status:** Approved (brainstorming) · **Date:** 2026-06-02 · **Component:** `crates/spur-notebook` (Tauri + Rust + React; kernel backend vendored from `ekzhang/jute` under `jute-notebook/src-tauri`)

## Overview & goals

The notebook today fully supports two kernels — **Python** (`python3`) and **JavaScript** (`deno`) — both with the SPUR **ports** system that lets cells pass tabular data to each other through Arrow files and a reactive DAG. **Rust** (`evcxr`) is *half-wired*: the `CodeType::Rust` enum variant and its kernelspec mapping exist, but every execution/provisioning path explicitly blocks it as *not yet supported*. **Go** does not exist anywhere.

This design adds:

1. **Rust** via the **evcxr** Jupyter kernel.
2. **Go** via the **gonb** kernel (`github.com/janpfeifer/gonb`).
3. **Full ports parity** for both — `spur.put` / `spur.get` participate in the same cross-language reactive DAG as Python/JS, reading and writing the byte-identical Arrow file + manifest format.

*"Bootstrap"* in this codebase means the kernel-provisioning routine (`ensure_*_kernelspec` in `kernel_provision.rs`) that prepares a kernel's runtime and registers a `kernel.json` into `~/.spur/jupyter/kernels/<name>/`. Making bootstrap *work with Rust and Go* means those `ensure_*` functions exist, are idempotent, and produce valid kernelspecs.

## Non-goals

- **Bundling toolchains.** Python bundles a `uv` sidecar because `uv` is small and downloads a managed interpreter. Full Rust and Go toolchains are hundreds of MB each plus build infrastructure — out of scope. We detect an existing `cargo`/`go` and install the kernel into it.
- **gophernotes.** The older Go kernel (built on the `gomacro` interpreter) is explicitly rejected in favor of gonb, which has modern Go-modules support, better Jupyter protocol compliance, and a clean path to pulling in the Arrow Go library for the ports shim.
- **Basic-execution-only mode.** We are doing full ports parity, not a minimal *run code and show output* REPL.

## The seven seams

Adding a language touches seven seams. Python/Deno already populate all seven; we fill the Rust row (half-stubbed) and add a Go row from scratch.

| # | Seam | File : line | Today | Add |
|---|------|-------------|-------|-----|
| 1 | Language enum + maps | `jute-notebook/src-tauri/src/backend/notebook.rs:251-258` (enum), `:261-267` (`kernelspec_for`), `~:269` (`code_type_for_spec`) | `CodeType { Python, Javascript, Rust }`; Rust→`"evcxr"` | Add `Go` variant; map `Go ↔ "gonb"` in both functions |
| 2 | Provisioning (*bootstrap*) | `jute-notebook/src-tauri/src/kernel_provision.rs` | `ensure_python3_kernelspec`, `ensure_deno_kernelspec` | `ensure_evcxr_kernelspec`, `ensure_gonb_kernelspec` |
| 3 | MCP provisioning gate | `src/mcp/tools/start_kernel.rs:~112` | `provisioning_target_for_spec`: `"evcxr" ⇒ NotYetSupported` | Add `Evcxr`, `Gonb` targets; dispatch to their `ensure_*` |
| 4 | DAG gate | `src/dag/engine.rs:684-699` | `reject_unsupported_kernel_specs` hard-rejects `spec_name == "evcxr"` | Stop rejecting `evcxr`/`gonb` (see next cell) |
| 5 | Ports shim | `jute-notebook/src-tauri/src/ports.rs:26` (`python_bootstrap`), `~:280` (`javascript_bootstrap`), `wrap_python_cell`/`wrap_js_cell` | Python + JS `_Spur`/`spur` preambles | `rust_bootstrap` + `wrap_rust_cell`; `go_bootstrap` + `wrap_go_cell` |
| 6 | Cell-wrap dispatch | `jute-notebook/src-tauri/src/commands.rs:1713` (`wrap_cell_for_kernel`) | `if spec_name == "deno" { js } else { python }` | 4-way match: `deno`→js, `evcxr`→rust, `gonb`→go, else→python |
| 7 | Frontend type | `jute-notebook/src/stores/notebook.ts:40-44` | `type SupportedKernelSpecName = "deno" \| "python3"` (unknown → `python3`) | Add `"evcxr" \| "gonb"`; update the fallback narrowing |

Seam 6 is the **single** execution-time wrap point — the DAG engine executes cells through this same `wrap_cell_for_kernel` path, so there is exactly one place to add language branches.

### DAG gate detail (seam 4)

`reject_unsupported_kernel_specs` (`src/dag/engine.rs:684-699`) currently filters `requirement.spec_name == "evcxr"` and raises `EngineError::UnsupportedKernelspec { spec_name: "evcxr".to_owned(), cell_ids }` — the literal `"evcxr"` is baked in twice. After this change both `evcxr` and `gonb` are supported, so the function no longer rejects them. **Keep the function** as the guard for genuinely-unknown spec names (so an unrecognized kernelspec still produces a clean structured error); it simply no longer treats `evcxr`/`gonb` as members of the unsupported set.

### Not a gate — error translators

`src/mcp/tools/notebook_run_cell.rs` and `src/mcp/tools/notebook_run_cascade.rs` are **not** independent language gates. They only *translate* `EngineError::UnsupportedKernelspec` into the MCP `kernelspec_not_supported` error payload. Once the engine (seam 4) stops rejecting `evcxr`/`gonb`, these paths go quiet for those specs automatically. Their unit tests construct `EngineError::UnsupportedKernelspec` by hand (with `spec_name: "evcxr"`) purely to assert the translation/`cell_ids` plumbing, so they remain valid and **must not be "fixed"** as part of this work.

## Provisioning design (seam 2 + 3)

Policy: **detect-and-install**, mirroring the existing `ensure_deno_kernelspec` pattern (check validity → resolve toolchain via env override then `$PATH` → install → **direct-write** `kernel.json` → re-validate). We reuse existing helpers in `kernel_provision.rs`: `kernelspec_is_valid`, `find_binary_on_path`, `existing_absolute_binary`, `binary_path_names`, `path_to_string`, and the `Error::KernelProvisionFailed { stage, cause }` type. Add per-kernel locks `EVCXR_KERNELSPEC_LOCK` and `GONB_KERNELSPEC_LOCK` (siblings of `PYTHON3_KERNELSPEC_LOCK` / `DENO_KERNELSPEC_LOCK`).

We **direct-write** the `kernel.json` (as Deno does) rather than calling `evcxr_jupyter --install` / `gonb --install` + `relocate_kernelspec`. The installers target the system Jupyter data dir and would force the relocate dance; direct-write lands the spec exactly where `environment::list_kernels` looks (`~/.spur/jupyter/kernels/<name>/`).

### `ensure_evcxr_kernelspec`

1. Short-circuit if `~/.spur/jupyter/kernels/evcxr/kernel.json` already passes `kernelspec_is_valid`.
2. Resolve `cargo`: `CARGO_PATH` env → `find_binary_on_path("cargo")`. If unresolved → `KernelProvisionFailed { stage: "cargo_path", cause: "could not resolve cargo from PATH; install Rust via https://rustup.rs or set CARGO_PATH" }`.
3. Run `cargo install --locked evcxr_jupyter` (idempotent). Failures → `stage: "evcxr_install"`.
4. Locate the built `evcxr_jupyter` (`~/.cargo/bin`, fallback `cargo install --list` / `$PATH`); validate it exists.
5. Direct-write `~/.spur/jupyter/kernels/evcxr/kernel.json`:
```json
{
  "argv": ["<abs evcxr_jupyter>", "--control_file", "{connection_file}"],
  "display_name": "Rust (SPUR)",
  "language": "rust"
}
```
6. Re-validate; error `stage: "kernelspec_validate"` if not.

### `ensure_gonb_kernelspec`

1. Short-circuit if `~/.spur/jupyter/kernels/gonb/kernel.json` already passes `kernelspec_is_valid`.
2. Resolve `go`: `GO_PATH` env → `find_binary_on_path("go")`. If unresolved → `KernelProvisionFailed { stage: "go_path", cause: "could not resolve go from PATH; install Go via https://go.dev/dl/ or set GO_PATH" }`.
3. Run `go install github.com/janpfeifer/gonb@latest` (GOBIN defaulting under the resolved Go env). Failures → `stage: "gonb_install"`.
4. Locate the built `gonb` (`$GOBIN`/`$GOPATH/bin`, fallback `$PATH`); validate it exists.
5. Direct-write `~/.spur/jupyter/kernels/gonb/kernel.json`:
```json
{
  "argv": ["<abs gonb>", "--kernel", "{connection_file}"],
  "display_name": "Go (SPUR)",
  "language": "go"
}
```
   (Match gonb's documented `kernel.json` argv; `{connection_file}` is substituted by the launcher.)
6. Re-validate; error `stage: "kernelspec_validate"` if not.

### MCP gate wiring (seam 3)

Extend `KernelspecProvisioningTarget` with `Evcxr` and `Gonb`. `provisioning_target_for_spec`: `"deno" ⇒ Deno`, `"evcxr" ⇒ Evcxr`, `"gonb" ⇒ Gonb`, `_ ⇒ Python3`. In `call()`, dispatch the new targets to `ensure_evcxr_kernelspec` / `ensure_gonb_kernelspec`; remove the `NotYetSupported` arm. Toolchain-missing errors flow through unchanged and read as actionable messages.

### Pinned versions

Add constants next to `MANAGED_KERNEL_DUCKDB_VERSION` (`kernel_provision.rs:16`) — consumed by the ports shims, not the install step:
```rust
/// Arrow Rust crate pulled into evcxr cells via `:dep` for the ports shim.
const EVCXR_ARROW_CRATE_VERSION: &str = "55";       // current-stable `arrow` crate
/// Arrow Go module pulled into gonb cells for the ports shim.
const GONB_ARROW_GO_MODULE: &str = "github.com/apache/arrow-go/v18";
```
The exact patch level is confirmed against what the kernel can actually compile during the implementation smoke test. The Go Arrow module moved from `github.com/apache/arrow/go/v18` → `github.com/apache/arrow-go/v18` around v18; the implementation validates the correct import path at smoke time.

## Ports parity contract (seam 5)

The Python `_Spur` shim (`ports.rs:26`) is the reference implementation. It:

- writes Arrow **IPC file-format** tables to `<notebook_port_root>/ports/<port>@v<version>.arrow`, where `notebook_port_root(path)` = `~/.spur/notebooks/<notebook_id>` and `notebook_id` = `nb-<blake3(path)[..24]>`;
- maintains `<notebook_port_root>/ports/manifest.json`, a versioned port registry, bumping `version` on each `put`;
- validates port names;
- emits a cell display payload under `PORT_MIME = "application/vnd.spur.port+json"`.

**The contract:** the Rust and Go shims must read/write the **byte-identical** Arrow IPC file format, the **same** `<port>@v<version>.arrow` path scheme, the **same** `manifest.json` schema + version-bump rule + port-name validation, and the **same** `PORT_MIME` display payload. This is what makes a port written by a Python cell readable by a Rust or Go cell, and vice-versa — the entire purpose of ports. Any divergence breaks cross-language interop silently.

Factor the shared format facts (the `ports/` subdir name, the `<port>@v<version>.arrow` filename scheme, the manifest JSON shape, `PORT_MIME`, and the port-name validation rule) so they are defined once and referenced by each language's bootstrap generator, rather than copy-pasted as string literals per shim.

### `rust_bootstrap` + `wrap_rust_cell`

Per-cell preamble (prepended to user code, like Python). Emits evcxr `:dep` directives first:
```
:dep arrow = "<EVCXR_ARROW_CRATE_VERSION>"
:dep serde_json = "1"
```
followed by a `spur` helper exposing `put(port, value)` / `get(port)` over `arrow::ipc::writer::FileWriter` / `arrow::ipc::reader::FileReader` against the same paths + manifest. evcxr caches `:dep` builds across cells in a session, so the compile cost is paid once.

### `go_bootstrap` + `wrap_go_cell`

Per-cell preamble for gonb. Emits the gonb module/import preamble pulling `<GONB_ARROW_GO_MODULE>` (`go get` semantics gonb expects) and defines `spur.Put` / `spur.Get` over the same Arrow IPC files + manifest JSON. gonb merges imports/`go get` directives across cells, so re-emitting per cell is safe.

### Cell-wrap dispatch (seam 6)

`wrap_cell_for_kernel(notebook_path, spec_name, code)` becomes a 4-way match on `spec_name`: `"deno"` → `wrap_js_cell`, `"evcxr"` → `wrap_rust_cell`, `"gonb"` → `wrap_go_cell`, else → `wrap_python_cell`. All four resolve `notebook_port_root(notebook_path)` identically.

### Known risk

The **first** `:dep arrow` build (evcxr) or `go get`/first-build (gonb) inside a freshly-started kernel is slow — tens of seconds — and **network-dependent** (crates.io / Go module proxy). This is the part most likely to need format/version tuning, and the implementation gates on a live smoke test. Subsequent cells in the same session reuse the cached build.

## Testing strategy

**Unit (Rust, no toolchain required):**

- `ensure_evcxr_kernelspec` / `ensure_gonb_kernelspec` with a recording/injected runner (same harness as the existing `kernel_provision` tests): toolchain present → writes a valid `kernel.json` with the expected argv; toolchain absent → returns the typed `KernelProvisionFailed { stage: "cargo_path" | "go_path", .. }`.
- `wrap_cell_for_kernel` 4-way dispatch routes each `spec_name` to the correct wrapper.
- `kernelspec_for` / `code_type_for_spec` round-trip including the new `Go ↔ "gonb"` mapping (and existing `Rust ↔ "evcxr"`).
- **Update existing "not supported" assertions:** the `evcxr`-rejection tests in `src/mcp/tools/start_kernel.rs` and `src/dag/engine.rs` flip from asserting rejection to asserting support/provisioning. The hand-constructed `UnsupportedKernelspec` translation tests in `notebook_run_cell.rs` / `notebook_run_cascade.rs` stay as-is.

**Integration (gated on toolchain availability, like the existing `PYTHON_PATH` e2e tests):**

- Cross-language port round-trip: start a Python kernel, `spur.put('t', <table>)`; start evcxr and gonb kernels, `spur.get('t')` in each, assert the table matches; then `put` from Rust/Go and `get` from Python. This is the acceptance gate validating the Arrow/manifest/MIME contract and the pinned Arrow versions end-to-end.
- A provisioning smoke test running the real `ensure_evcxr_kernelspec` / `ensure_gonb_kernelspec` when `cargo` / `go` are present, asserting a startable kernel.

## Open questions

- **Arrow patch versions.** `EVCXR_ARROW_CRATE_VERSION` (`"55"`) and the `arrow-go/v18` import path are pinned at the current-stable line; the exact buildable patch is confirmed at the implementation smoke test against the actual installed toolchains. If the smoke test reveals a version the kernel can't compile, the constant is adjusted — no design change.
- **`display_name` strings.** `"Rust (SPUR)"` / `"Go (SPUR)"` match the `"Python 3 (SPUR)"` convention; trivial to rename if product wants different labels.

In [ ]:
# SPUR datasource setup cell v1
# This cell is managed by SPUR. Re-run it after datasource changes.
import duckdb

_SPUR_DUCKDB_EXTENSION_PATH = "/Users/kevintruong/.spur/extensions/spur_rest.duckdb_extension"
_SPUR_DUCKDB_EXTENSION_SQL = _SPUR_DUCKDB_EXTENSION_PATH.replace("'", "''")

if "_SPUR_DUCKDB_CONNECTION" not in globals():
    _SPUR_DUCKDB_CONNECTION = duckdb.connect(
        database=":memory:",
        config={"allow_unsigned_extensions": "true"},
    )

duckdb.set_default_connection(_SPUR_DUCKDB_CONNECTION)
duckdb.sql(f"LOAD '{_SPUR_DUCKDB_EXTENSION_SQL}'")

